# Neutralizer

A 1 MeV, 10 A deuterium negative-ion beam crosses one channel of a gas neutralizer, 3 m long
and 10 cm wide, as in ITER's injectors. On the gas, and on the weak plasma it makes there
itself, the beam changes charge state: about 57 % of it leaves as atoms, the neutral beam that
heats the plasma.

The plasma is a global model of the channel from [carafe](https://github.com/max-models/carafe)
(the `neutralizer` extra), fed by the beam's stripped and secondary electrons. Alone, that
model treats the beam as a line integral along the axis. `beamtracer.neutralizer.NeutralizerChannel`
traces the beam instead: `BeamOptics` trajectories through the channel in $(x, y)$, each
carrying its charge-state fractions on the gas and the plasma, while the plasma takes its
sources from the traced paths. The two are iterated to a fixed point.

The cross sections and rates are illustrative placeholders, not data: the figures show what
the coupled models compute, not what a neutralizer achieves.

In [ ]:
import warnings

import numpy as np
import plasma_plots as pp
from carafe.cr import deuterium_neutralizer

from beamtracer.neutralizer import NeutralizerChannel

BLUE, ORANGE, GREEN, GREY = "C0", "C1", "C2", "0.6"
pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## The channel

`deuterium_neutralizer` is the channel with its gas fed at $6 \times 10^{21}$ molecules per
second, near the gas's best neutral fraction, and OpenCR's plasma rates; `cusp=None` makes it
a gas neutralizer (with magnetic cusps on the walls it would be the plasma neutralizer studied
for DEMO). The beam enters 8 cm wide, with an rms divergence that the scan below varies.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # the placeholder beam data warn
    neutralizer = deuterium_neutralizer(gas_inflow=6.0e21, cusp=None)


def channel(divergence: float) -> NeutralizerChannel:
    # the traced channel for a beam of rms divergence [rad]
    return NeutralizerChannel(
        neutralizer, beam_width=0.08, divergence=divergence, num_particles=200
    )


result = channel(0.01).solve()
print(
    f"{'converged' if result.converged else 'not converged'} in {result.iterations} iterations; "
    f"electron density {result.plasma.electron_density:.2e} m^-3"
)
for state, share in result.exit_fractions.items():
    print(f"{state:>2} at the exit: {share:.3f} of the incoming particles")
print(f"transmission {result.transmission:.3f}")

## The beam through the channel

At 10 mrad rms the beam spreads by about 3 cm per side over the 3 m, and its edges reach the
walls of the 10 cm channel before the exit.

In [ ]:
with pp.figure(title="The beam through the channel, 10 mrad rms") as fig:
    ax = fig[0]
    for y in (-0.05, 0.05):
        ax.plot([0.0, 3.0], [y * 1e3] * 2, color="k", lw=3)
    for path in result.model.trajectories:
        ax.plot(path[:, 0], path[:, 1] * 1e3, color=BLUE, lw=0.6)
    ax.set_xlabel("$x$ [m]")
    ax.set_ylabel("$y$ [mm]")

## The charge states

Along the channel, the fractions of the global model on the axis: the D$^-$ is stripped to
atoms, the atoms are ionized in turn, and the neutral fraction levels off where the two
balance. At the exit, the traced beam: what leaves in each state, as a share of the particles
that entered.

In [ ]:
axis = result.axis
with pp.figure(title="Charge states along the channel") as fig:
    ax = fig[0]
    for k, (state, label, color) in enumerate(
        (("-", "D$^-$", BLUE), ("0", "D", ORANGE), ("+", "D$^+$", GREEN))
    ):
        ax.plot(axis.position, axis.fractions[:, k], color=color, label=label)
        ax.plot([axis.position[-1]], [result.exit_fractions[state]], "o", color=color)
    ax.set_xlabel("$x$ [m]  (dots: the traced beam at the exit)")
    ax.set_ylabel("fraction of the incoming particles")
    ax.legend()

## Divergence and the walls

The same channel for beams of 0 to 15 mrad. A parallel beam gives the axis model's neutral
fraction; a diverging one loses its wide angles to the walls.

In [ ]:
divergences = (0.0, 0.003, 0.006, 0.01, 0.015)
runs = [channel(d).solve() for d in divergences]
for d, run in zip(divergences, runs, strict=True):
    print(
        f"{d * 1e3:4.1f} mrad: transmission {run.transmission:.3f}, "
        f"neutral fraction {run.neutral_fraction:.3f} (axis {run.axis.neutral_fraction:.3f}), "
        f"neutrals leave at {run.neutral_divergence * 1e3:.1f} mrad"
    )

In [ ]:
mrad = np.array(divergences) * 1e3
with pp.figure(title="What the walls take") as fig:
    ax = fig[0]
    transmission = np.array([run.transmission for run in runs])
    neutral = np.array([run.neutral_fraction for run in runs])
    ax.plot(mrad, transmission, "o:", color=GREY, label="transmission (all states)")
    ax.plot(mrad, neutral, "o-", color=ORANGE, label="neutral fraction, traced")
    ax.plot(mrad, [run.axis.neutral_fraction for run in runs], "--", color="k",
            label="neutral fraction, along the axis")  # fmt: skip
    ax.plot(mrad, neutral / transmission, "s-", color=BLUE,
            label="neutrals in the transmitted beam")  # fmt: skip
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("rms divergence at the entrance [mrad]")
    ax.set_ylabel("share of the incoming particles")
    ax.legend()

At 10 mrad a fifth of the beam hits the walls and the neutral fraction falls from 0.57 to
0.47, but the neutral share of the beam that leaves stays at 0.57: every transmitted particle
crosses the same gas over the same length, and the plasma strips little, so the loss is all
geometry. The neutrals that leave diverge less than the beam that entered, since the wide
angles are the ones lost. The neutrals go on into the [beamline](/beamtracer/beamline/).